In [12]:
import pandas as pd, numpy as np
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def load_log(logdir: str):
    """logdir: Log's root folder
    Returns: [Pandas Dataframe of Robots Logs, Dict of TaskMap, Tuple with grid size]
        """
    #loads log to pandas
    bots_path = os.path.join(logdir, "robots_log.jsonl")
    with open(bots_path, "r") as f:
        df = pd.read_json(f, lines=True)
    df["pos"] = df["pos"].apply(tuple)
    
    #loads taskMap to dict
    taskmap_path = os.path.join(logdir, "grid_taskMap.json")
    with open(taskmap_path, "r") as f:
        taskMap = json.load(f)
        
    #get grid size
    positions = [
        tuple(int(v) for v in key.strip("()").split(","))
        for key in taskMap.keys()
    ]
    cols = max(x for x, y in positions) + 1
    rows = max(y for x, y in positions) + 1

    return df, taskMap, (rows, cols)

def count_blocks(shape: list) -> dict:
    shape = np.array(shape)
    blocks = {}
    blocks["empty"]     = int(np.sum(shape == 0))
    blocks["rigid"]     = int(np.sum(shape == 1))
    blocks["soft"]      = int(np.sum(shape == 2))
    blocks["h_act"]     = int(np.sum(shape == 3))
    blocks["v_act"]     = int(np.sum(shape == 4))
    blocks["actuators"] = blocks["h_act"] + blocks["v_act"]
    blocks["n_blocks"]  = blocks["rigid"] + blocks["soft"] + blocks["h_act"] + blocks["v_act"]
    return blocks

def build_actuator_maps(df: pd.DataFrame, rows:int, cols:int):
    """
    Builds a matriz of (gens, rows, cols) with the numbers of actuators;
    Returns matrix and max/min values of actuators
    """
    #grabs all present generations
    generations = sorted(df["gen"].unique())
    n_gens = len(generations)

    #starts matrix with -1 to indicate stuff that was not found in dataframe
    matrix = np.full((n_gens, rows, cols), fill_value=-1, dtype=float)

    #iterates throught generations and builds matrix
    for gen in generations:
        #get robots of this gen
        genBots = df[df["gen"]==gen]

        #write each bot of this gen
        for _, row in genBots.iterrows():
            x, y = row["pos"]
            matrix[gen, y, x] = row["actuators"]

    missing = np.sum(matrix == -1)
    if missing > 0: print("Missing values in matrix!")

    minValue = matrix.min()
    maxValue = matrix.max()
    
    return matrix, generations, maxValue, minValue

def build_task_overlay(taskMap:dict, rows:int, cols:int):
    """Returns a matrix assigning an int for each task in the taskMap
    """
    taskNames = sorted(set(taskMap.values())) #sort alphabetically to guarantee same order in different executions
    overlayMatrix = np.zeros((rows, cols), dtype=int)
        
    for key, task in taskMap.items():
        x, y = key.strip("()").split(",")
        overlayMatrix[int(y),int(x)] = taskNames.index(task)
        
    return overlayMatrix, taskNames    

def render_generation_map(
    targetMatrix: np.ndarray, #matrix with relevant data; ex: number of actuators for each cell in the grid
    taskMatrix: np.ndarray,   #matrix that indicates which task is in which cell
    taskNames: list,          #task names in the correct order - indexes here indicate tasks in taskMatrix
    minValue: float,
    maxValue: float,          #min and max value of targetMatrix, used for heatmap.
    gen: int,                 #generation of this map
    taskColors: list[str],    #colors of each task in taskNames (for rectangle) 
    figSize: tuple = (10,10) ) -> np.ndarray:
    
    plt.close("all")
    rows, cols = targetMatrix.shape
    fig, ax = plt.subplots(figsize = figSize)

    #set heatmap with max and min values
    im = ax.imshow(
        targetMatrix,
        cmap='seismic',
        vmin=minValue,
        vmax=maxValue,
        aspect="equal"
    )
    
    #print borders for different tasks
    for y in range(rows):
        for x in range(cols):
            taskIndex = taskMatrix[y, x]
            color = taskColors[taskIndex]
            
            rect = patches.Rectangle(
                (x - 0.5, y - 0.5), 
                1, 1,                  
                linewidth=3,
                edgecolor=color,
                facecolor="none")
            ax.add_patch(rect)

    #value inside cell
    for y in range(rows):
        for x in range(cols):
            ax.text(
                x, y,
                f"{int(targetMatrix[y, x])}",
                ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="black")
            
    #color bar and title
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of Actuators", fontsize=10)

    #task subtitles
    legendElements = [
        patches.Patch(edgecolor=taskColors[i], facecolor="none",
                      linewidth=3, label=taskNames[i].split(".")[-1])
        for i in range(len(taskNames))]
    ax.legend(handles=legendElements, loc="upper left",
              bbox_to_anchor=(1.15, 1.0), fontsize=9)

    ax.set_title(f"Generation {gen}", fontsize=13)
    ax.axis("off")

    fig.tight_layout()
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())  # shape (h, w, 4) — RGBA
    frame = frame[:, :, :3]                        # descarta canal alpha → RGB
    plt.close(fig)
    return frame

In [13]:
logdir = "../log/tests_CGA_03231847_no_toroid"  # Simulation to gather data from
df, taskMap, gridSize = load_log(logdir)
rows, cols = gridSize

#added coluns with robot's data
newCols = df["shape"].apply(count_blocks).apply(pd.Series)
df = pd.concat([df, newCols], axis=1)

# print(f"Grid size: {rows}×{cols} | Generations: {df['gen'].max()} | Entries: {len(df)}")
# print(df.head())

In [14]:
plt.close("all")
matrix, generations, maxValue, minValue = build_actuator_maps(df, rows, cols)
overlayMatrix, taskNames = build_task_overlay(taskMap,rows,cols)
taskColors = ["yellow","green"]

# gens = [0, 25, 60, 75, 100]
# for gen in gens:
#     fig = render_generation_map(matrix[gen], overlayMatrix, taskNames, minValue, maxValue, gen, ["yellow","green"], (10,10))
#     display(fig)
#     plt.close(fig)

import imageio
plt.close("all")
frames = []

for g_idx, gen in enumerate(generations):
    frame = render_generation_map(
        matrix[g_idx], overlayMatrix, taskNames,
        minValue, maxValue, gen, taskColors)
    frames.append(frame)
    
    if g_idx % 10 == 0:
        print(f"Frame {g_idx}/{len(generations)} gerado...")

# Salva o GIF
output_path = os.path.join(logdir, "heatmap_actuators.gif")
imageio.mimsave(output_path, frames, duration=150)  # 150ms por frame
print(f"GIF salvo em: {output_path}")


Frame 0/101 gerado...
Frame 10/101 gerado...
Frame 20/101 gerado...
Frame 30/101 gerado...
Frame 40/101 gerado...
Frame 50/101 gerado...
Frame 60/101 gerado...
Frame 70/101 gerado...
Frame 80/101 gerado...
Frame 90/101 gerado...
Frame 100/101 gerado...
GIF salvo em: ../log/tests_CGA_03231847_no_toroid/heatmap_actuators.gif
